# Ableton AI Chatbot

In [ ]:
# Import Required Libraries
import os
import re
import gradio as gr
import ollama
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

## Configuration

In [ ]:
# Model Configuration
EMBED_MODEL = "nomic-embed-text"
OLLAMA_MODEL = "llama3.2"

# Vector Store Path
DEFAULT_CHROMA_PATH = os.path.expanduser("~/ableton_manual_vectors")
CHROMA_PATH = os.environ.get("CHROMA_PATH", DEFAULT_CHROMA_PATH)

In [ ]:
# Open AI configuration
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# Initialize OpenAI client
openai = OpenAI()

# Model Configuration
OPENAI_MODEL = "gpt-4o-mini"  # OpenAI model for chat

## Load Vector Store

In [ ]:
# Load the pre-built vector store
embedding = OllamaEmbeddings(model=EMBED_MODEL)
vectordb = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding)

# Verify vector store loaded successfully
count = vectordb._collection.count()
print(f"✅ Loaded vector store with {count:,} vectors.")

### Print a few examples of stored content with metadata

In [ ]:
print("\n" + "="*80)
print("SAMPLE DOCUMENTS IN VECTOR STORE:")
print("="*80)

collection = vectordb._collection
samples = collection.get(limit=5, include=['documents', 'metadatas'])

for i, (doc, metadata) in enumerate(zip(samples['documents'], samples['metadatas']), 1):
    print(f"\n--- Sample {i} ---")
    print(f"Product: {metadata.get('product', 'N/A')}")
    print(f"Source: {metadata.get('source', 'N/A')}")
    print(f"Content preview (first 500 chars):")
    print(doc[:500] + "..." if len(doc) > 500 else doc)
    print()

print("="*80)

## Helper Functions

In [ ]:
def build_system_prompt(context_text: str) -> str:
    """Build the system prompt with retrieved context."""
    return f"""You are **Ableton Assistant**.

# SCOPE
- Answer **only** about Ableton products and official manuals (Live, Push, Move, devices, workflows).
- If a user asks about **buying, pricing, sales, or store information**, redirect them to: https://www.ableton.com/shop/
- Answer accurately using the Ableton manuals and technical context provided.
- If the question is outside this scope or the answer is not in the provided context, say you don't know.

# SOURCING
- Use **only** the provided CONTEXT as the authoritative source.
- If the context is insufficient, say so and suggest a relevant manual section to consult.

# STYLE
- Be concise, accurate, and practical.
- When helpful, mention the product or section inferred from metadata.

# CONTEXT (authoritative)
{context_text}
"""

In [ ]:
# Input sanitization patterns
FORBIDDEN = [
    r"\bignore\b",
    r"\bpretend\b",
    r"\bbehave\s+as\b",
    r"\bchange\s+(role|persona|character)\b",
    r"\byou\s+are\b",
    r"\bimpersonate\b",
    r"\bhate\b",
    r"\bpolitician\b",
    r"\bpolitics\b",
    r"\bracist\b",
    r"\bkill\b",
    r"\btrump\b",
    r"\bev(il|ildo)\b",
]

def sanitize_user_input(text: str) -> str:
    """
    Return a safe version of user input.
    If it contains jailbreak / unsafe language, return a friendly block message.
    """
    for pat in FORBIDDEN:
        if re.search(pat, text, re.IGNORECASE):
            print(
                f"============================\n"
                f"🚫 Blocked pattern matched: {pat}\n"
                f"============================"
            )
            return "I prefer not to talk about this topic. Do you want to learn more about Ableton products instead?"
    return text

In [ ]:
def detect_product_from_query(query: str) -> str | None:
    """Detect which Ableton product is mentioned in the query."""
    q = query.lower()
    if "push" in q:
        return "push"
    if "move" in q:
        return "move"
    if "live" in q:
        return "live"
    return None

In [ ]:
def retrieve_context(query: str, top_k: int = 12, final_k: int = 6, max_chars: int = 5000) -> str:
    """
    Retrieve a richer, deduplicated Ableton-manual context.
    
    Args:
        query: User's question
        top_k: Number of documents to retrieve initially
        final_k: Number of top documents to keep after scoring
        max_chars: Maximum characters to include in context
    
    Returns:
        Concatenated context string from relevant manual sections
    """
    # Detect product and apply filter if found
    product = detect_product_from_query(query)
    filter_args = {"product": product} if product else None

    # Retrieve documents with similarity scores
    results = vectordb.similarity_search_with_score(query, k=top_k, filter=filter_args)

    # Sort by ascending distance (lower is better)
    results = sorted(results, key=lambda x: x[1])
    docs = [d for d, _ in results[:final_k]]

    # Deduplicate by normalized text
    seen = set()
    unique = []
    for d in docs:
        text = d.page_content.strip()
        key = " ".join(text.lower().split())
        if key not in seen:
            seen.add(key)
            unique.append(text)

    # Concatenate and truncate if too long
    context = "\n---\n".join(unique)
    if len(context) > max_chars:
        context = context[:max_chars]

    return context

In [ ]:
def content_to_str(content) -> str:
    # Gradio may provide: str OR list-of-parts like [{"type":"text","text":"..."}]
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            part.get("text", "")
            for part in content
            if isinstance(part, dict) and part.get("type") == "text"
        )
    return "" if content is None else str(content)


def messages_for_llm(msgs):
    return [{"role": m["role"], "content": content_to_str(m.get("content"))} for m in msgs]


## Chat Function (Ollama and OpenAI)

In [ ]:
DEBUG = True

def stream_ollama(messages):
    """Stream response from Ollama."""
    messages = messages_for_llm(messages)
    stream = ollama.chat(model=OLLAMA_MODEL, messages=messages, stream=True)
    
    partial = ""
    for chunk in stream:
        delta = chunk.get("message", {}).get("content", "")
        if delta:
            partial += delta
            yield partial


def stream_openai(messages):
    """Stream response from OpenAI."""
    messages = messages_for_llm(messages)
    stream = openai.chat.completions.create(
        model=OPENAI_MODEL,
        messages=messages,
        stream=True,
        temperature=0.7
    )
    
    partial = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        if delta:
            partial += delta
            yield partial


def chat_with_ableton_generic(user_message, history, stream_fn):
    """
    Generic chat function that works with any streaming LLM.
    
    Args:
        user_message: The user's input message
        history: Chat history (list of message dictionaries)
        stream_fn: Function that streams responses from the LLM
    
    Yields:
        Streamed response chunks
    """
    # Sanitize input
    cleaned_input = sanitize_user_input(user_message)
    if cleaned_input != user_message:
        yield cleaned_input
        return

    # Retrieve relevant context from vector store
    context_text = retrieve_context(user_message)
    if DEBUG:
        print("\n============ 🔎 Retrieved context from manuals ============")
        print(context_text)

    # Build system prompt with context
    system_prompt = build_system_prompt(context_text)

    # Prepare messages for the LLM
    messages = [{"role": "system", "content": system_prompt}]
    
    # Add history
    for msg in history:
        role = msg["role"]
        content = msg.get("content", "")
        if isinstance(content, str):
            messages.append({"role": role, "content": content})
    
    # Add current user message
    messages.append({"role": "user", "content": user_message})

    # Stream response using the provided function
    yield from stream_fn(messages)


# Specific implementations
def chat_with_ableton_ollama(user_message, history=[]):
    """Ollama version of the chatbot."""
    yield from chat_with_ableton_generic(user_message, history, stream_ollama)


def chat_with_ableton_openai(user_message, history):
    """OpenAI version of the chatbot."""
    yield from chat_with_ableton_generic(user_message, history, stream_openai)

## UI 1: Basic Chatbot (Ollama)

In [ ]:
# Basic UI without greeting
demo_basic = gr.ChatInterface(
    fn=chat_with_ableton_ollama,
    title="Ableton Assistant (Ollama)",
    description="AI assistant about Ableton products - Live, Push, or Move.",
    examples=[
        "How do I quantize audio in Ableton Live?",
        "How do I record MIDI notes in Ableton Live?",
        "How do I connect Push to my computer?",
        "How do I record a drum pattern using Push pads?",
    ],
)

demo_basic.launch(inbrowser=False)

## UI 2: Chatbot with Greeting (Ollama)

In [ ]:
# UI with greeting message
greeting = [
    {
        "role": "assistant",
        "content": (
            "👋 Hi there! I'm your Ableton Manual Assistant.\n"
            "Ask me anything about Ableton Live, Push, or Move — "
            "I'll do my best to answer your question."
        ),
    }
]

bot = gr.Chatbot(value=greeting)

demo_greeting = gr.ChatInterface(
    fn=chat_with_ableton_ollama,
    title="Ableton Assistant (Ollama)",
    description="Ask about Ableton Live, Push, or Move manuals.",
    chatbot=bot,
)

demo_greeting.launch(inbrowser=True)

## UI 3: Chatbot with Greeting (OpenAI)

In [ ]:
# Basic UI with greeting
greeting = [
    {
        "role": "assistant",
        "content": (
            "👋 Hi there! I'm your Ableton Manual Assistant (powered by OpenAI GPT-4o-mini).\n"
            "Ask me anything about Ableton Live, Push, or Move — "
            "I'll do my best to answer your question using the official manuals."
        ),
    }
]

bot = gr.Chatbot(value=greeting)

demo_greeting = gr.ChatInterface(
    fn=chat_with_ableton_openai,
    title="Ableton Assistant (OpenAI)",
    description="Ask about Ableton Live, Push, or Move manuals.",
    chatbot=bot,
)

demo_greeting.launch(inbrowser=True)